# Tests of Completeness, Representtiveness

Replay of the V1 example notebook with the new API V2 proposed in V1.2.3
- Previous code will be commented when needed
- replacement included 

In [ ]:
# Common Libraries
import pandas as pd
import numpy as np

## Tests on challenge welding data

This dataset is described on the website of challenge welding : https://confianceai.github.io/Welding-Quality-Detection-Challenge/


In [ ]:
# upload data 
# Dataset path
path = "datasets/challenge_welding.csv"
data = pd.read_csv(path, sep=",")
#print("Data info:\n", data.info())
# print("The first 5 lines:\n", data.head())
data

# Completeness 
you can see also the file: *dqm/completeness/main.py*

In [ ]:
## Specific Libraries dqml-ml V1
from dqm.completeness.metric import DataCompleteness

## Specific Libraries dqml-ml V2
from dqm_ml_core import CompletenessProcessor
from dqm_ml_core import  MetricRunner

In [ ]:
# Compute completeness on the whole dataset
completeness_evaluator = DataCompleteness()
overall_score = completeness_evaluator.completeness_tabular(data)
print("V1 : ", overall_score)

runner = MetricRunner()

metric = CompletenessProcessor(config={"include_per_column" : False})
metrics_values = runner.run(data, [metric])
print("V2 : ", metrics_values)

In [ ]:
# Compute completeness on a single column
column_score = completeness_evaluator.data_completion(data['blur_level'])

# Print the results
print(f'Overall Data Completeness Score: {overall_score}')
print(f'Completeness Score for Column: {column_score}')

# With API V2 - configure column and metric metric name in outdict
metric = CompletenessProcessor(config={"input_columns" : ["blur_level"]})
metrics_values = runner.run(data, [metric])

print(f'Overall Data Completeness Score: ', metrics_values['completeness_overall'])
print(f'Completeness Score for Column: ', metrics_values['completeness_blur_level'])

# Representativeness

In [ ]:
## Specific Libraries
from dqm.representativeness.metric import DistributionAnalyzer

In [ ]:
var = data["blur_level"] # you can choose another variable from the data. 
mean = np.mean(var)
std = np.std(var)
# data = pd.Series(np.random.normal(0, 1, 1000))
print(mean,std)

In [ ]:
# Parameters for analysis
bins = 20
distribution = 'normal'

# Instantiation of DistributionAnalyzer
analyzer = DistributionAnalyzer(var, bins, distribution)

In [ ]:
# Using the method chisquare_test
pvalue, _ = analyzer.chisquare_test()
print(f"Chi-Square Test: p-value = {pvalue}")

In [ ]:
# Using the method chisquare_test for V3
## Specific Libraries dqml-ml V2
from dqm_ml_core import RepresentativenessProcessor

bins = 20
distribution = 'normal'

metric = RepresentativenessProcessor(config={"metrics" : ["chi-square"], "input_columns" : ["blur_level"], "bins": 20, "distribution" : 'normal'})
metrics_values = runner.run(data, [metric])
print(metrics_values)

print(f"Chi-Square Test: p-value = ", metrics_values["chi-square_blur_level_p_value"])

#pvalue, intervals_frequencies = analyzer.chisquare_test()
#print(f"Chi-Square Test: p-value = {pvalue}")
#print(f"Chi-Square Test: discretized intervalls are \n = {intervals_frequencies}")

In [ ]:
# Using the method kolmogorov
ks_pvalue = analyzer.kolmogorov(mean,std)
print(f"Kolmogorov-Smirnov Test: p-value = {ks_pvalue}")

In [ ]:
metric = RepresentativenessProcessor(config={"metrics" : ["kolmogorov-smirnov"], "input_columns" : ["blur_level"], "bins": 20, "distribution" : 'normal'})
metrics_values = runner.run(data, [metric])
print(metrics_values)

print(f"Kolmogorov-Smirnov Test: p-value = ", metrics_values["kolmogorov-smirnov_blur_level_p_value"])

In [ ]:
# Using the method shannon_entropy
entropy = analyzer.shannon_entropy()
print(f"Shannon Entropy: {entropy}")

In [ ]:
metric = RepresentativenessProcessor(config={"metrics" : ["shannon-entropy"], "input_columns" : ["blur_level"], "bins": 20, "distribution" : 'normal'})
metrics_values = runner.run(data, [metric])
print(metrics_values)

print(f"Shannon Entropy Test: p-value = ", metrics_values["shannon-entropy_blur_level_entropy"])

In [ ]:
# Using the method grte
grte_result, intervals_discretized = analyzer.grte()
print(f"GRTE: {grte_result}")
#print(f"GRTE discretized intervalls are \n = {intervals_discretized}") # uncomment to display data discretization

In [ ]:
metric = RepresentativenessProcessor(config={"metrics" : ["grte"], "input_columns" : ["blur_level"], "bins": 20, "distribution" : 'normal'})
metrics_values = runner.run(data, [metric])
print(metrics_values)

print(f"GRTE = ", metrics_values["grte_blur_level_grte_value"])

# We can compute all metrics at once

In [ ]:
metric = RepresentativenessProcessor(config={"input_columns" : ["blur_level"], "bins": 20, "distribution" : 'normal'})
metrics_values = runner.run(data, [metric])
print(metrics_values)